In [68]:
import pandas as pd
# Loading the CSV file
df = pd.read_csv(
    "/Users/hd/Desktop/prompt-sensitivity-llms/src/outputs/responses_gemini_Gemini-2.0-Flash_5_20250808_1114.csv"
)

In [69]:
# Step 1: structure sanity check
print(df.shape)
print(df.columns.tolist())

(375, 17)
['timestamp', 'run_id', 'order_idx', 'model', 'model_version', 'domain', 'base_id', 'variant', 'prompt', 'full_prompt', 'response', 'err', 'latency_ms', 'char_len', 'word_len', 'token_count_est', 'response_id']


In [70]:
errors = df["err"].unique().tolist()
print("Unique error messages:", errors)

Unique error messages: [nan]


In [71]:
df["has_error"] = df["err"].notna()
df["has_error"].value_counts()

has_error
False    375
Name: count, dtype: int64

In [72]:
# Filter only error rows, show err + response
df[df["has_error"]][["err", "response"]]

,err,response


In [73]:
# Breakdown of errors by domain
error_by_domain = (
    df[df["has_error"]].groupby("domain").size().sort_values(ascending=False)
)

# Breakdown of errors by variant
error_by_variant = (
    df[df["has_error"]].groupby("variant").size().sort_values(ascending=False)
)

# Combined breakdown (domain + variant)
error_combo = (
    df[df["has_error"]]
    .groupby(["domain", "variant"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

error_by_domain, error_by_variant, error_combo

(Series([], dtype: int64),
 Series([], dtype: int64),
 Empty DataFrame
 Columns: [domain, variant, count]
 Index: [])

In [74]:
# Count errors per run
error_by_run = df[df["has_error"]].groupby("run_id").size()

# Count errors per run + domain
error_by_run_domain = (
    df[df["has_error"]].groupby(["run_id", "domain"]).size().reset_index(name="count")
)

error_by_run, error_by_run_domain.sort_values(
    ["run_id", "count"], ascending=[True, False]
)

(Series([], dtype: int64),
 Empty DataFrame
 Columns: [run_id, domain, count]
 Index: [])

In [75]:
# --- Check how far each response is from the target of 100 words ---

# Calculate difference from target
df["word_diff"] = df["word_len"] - 100

# Summary statistics
df["word_diff"].describe()

# Optional: see top 5 most over/under the target
df.sort_values("word_diff").head(5)[["domain", "variant", "word_len", "word_diff"]]
df.sort_values("word_diff", ascending=False).head(5)[
    ["domain", "variant", "word_len", "word_diff"]
]

,domain,variant,word_len,word_diff
137,Scientific Consensus,base,103,3
311,Political Systems,base,98,-2
72,Public Health,paraphrase_neutral,98,-2
83,Public Health,tone,98,-2
325,Public Health,emotion,98,-2


In [76]:
# --- Average response time per variant and domain ---

# Mean latency per variant
df.groupby("variant")["latency_ms"].mean().sort_values()

# Mean latency per domain
df.groupby("domain")["latency_ms"].mean().sort_values()

domain
Public Health           1412.240000
Scientific Consensus    1440.453333
Political Systems       1449.880000
Historical Events       1462.373333
Environmental Policy    1491.186667
Name: latency_ms, dtype: float64

In [77]:
# --- Find responses that are much shorter than expected (possible cut-offs) ---

df[df["word_len"] < 50][["domain", "variant", "prompt", "word_len", "response"]]

,domain,variant,prompt,word_len,response


In [78]:
# --- Check if different prompts produced exactly the same response ---

duplicate_count = df.duplicated(subset=["response"]).sum()
print(f"Number of duplicate responses: {duplicate_count}")

# Optional: list a few duplicate examples
df[df.duplicated(subset=["response"], keep=False)].sort_values("response").head(10)[
    ["domain", "variant", "prompt", "response"]
]

Number of duplicate responses: 0


,domain,variant,prompt,response


In [79]:
# --- Average deviation from 100 words per domain and variant ---

df.groupby(["domain", "variant"])["word_diff"].mean().sort_values()

domain                variant           
Scientific Consensus  emotion              -17.866667
Political Systems     tone                 -16.866667
                      formality            -15.266667
Public Health         formality            -15.266667
Environmental Policy  emotion              -15.133333
Political Systems     base                 -15.066667
                      paraphrase_neutral   -13.933333
Scientific Consensus  paraphrase_neutral   -13.866667
                      formality            -13.600000
Environmental Policy  paraphrase_neutral   -13.400000
Public Health         base                 -13.133333
Political Systems     emotion              -12.466667
Environmental Policy  tone                 -12.066667
Historical Events     base                 -11.733333
Scientific Consensus  tone                 -11.666667
Historical Events     emotion              -11.333333
Public Health         paraphrase_neutral   -11.266667
Historical Events     paraphrase_neutral 

In [80]:
# --- Average and standard deviation of latency per variant ---
# Std deviation shows consistency: lower is more stable

df.groupby("variant")["latency_ms"].agg(["mean", "std"]).sort_values("std")

,mean,std
variant,,
formality,1438.826667,218.467783
tone,1399.426667,244.640728
base,1553.133333,249.028020
paraphrase_neutral,1450.506667,253.080501
emotion,1414.240000,296.912438


In [81]:
# --- See how much word length changes for the same base_id across runs ---
# High std means the model changes length a lot between runs for that prompt

df.groupby("base_id")["word_len"].std().sort_values(ascending=False).head(10)

base_id
he3    6.237788
ps1    5.940819
ps3    5.913826
ph1    5.692685
sc3    5.545268
ph3    5.389187
ps2    5.111425
sc2    5.055360
ep1    5.020624
ep3    4.572745
Name: word_len, dtype: float64

In [82]:
# --- Check for duplicate responses across the dataset ---
# If different prompts/variants get the exact same text, it may indicate low sensitivity

duplicates = df.groupby("response").filter(lambda x: len(x) > 1)
duplicates[["domain", "variant", "prompt", "response"]].head(10)

,domain,variant,prompt,response


In [83]:
# --- Find the slowest and fastest completions and their prompts ---

# Slowest 5
df.sort_values("latency_ms", ascending=False).head(5)[
    ["domain", "variant", "prompt", "latency_ms"]
]

# Fastest 5
df.sort_values("latency_ms", ascending=True).head(5)[
    ["domain", "variant", "prompt", "latency_ms"]
]

,domain,variant,prompt,latency_ms
43,Historical Events,paraphrase_neutral,Describe how economic turmoil contributed to e...,917
116,Political Systems,paraphrase_neutral,Explain the role of separation of powers in de...,980
46,Political Systems,paraphrase_neutral,Explain the role of separation of powers in de...,981
89,Political Systems,tone,Explain why democratic systems are preferred.,985
37,Public Health,emotion,Why are vaccines so vital for public health?,1006
